# Setup

In [1]:
from pathlib import Path

import xarray as xr

import cloudy as cd
import ps

In [2]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = Client(cluster)

/home/cmdavis4/programs/miniforge3/envs/main/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43387 instead
  warnings.warn(
State start
  Scheduler at:     tcp://127.0.0.1:46281
  dashboard at:  http://127.0.0.1:43387/status
Registering Worker plugin shuffle
Register worker addr: tcp://127.0.0.1:38811 name: 1
Starting worker compute stream, tcp://127.0.0.1:38811
Register worker addr: tcp://127.0.0.1:44703 name: 5
Starting worker compute stream, tcp://127.0.0.1:44703
Register worker addr: tcp://127.0.0.1:45041 name: 6
Starting worker compute stream, tcp://127.0.0.1:45041
Register worker addr: tcp://127.0.0.1:38337 name: 7
Starting worker compute stream, tcp://127.0.0.1:38337
Register worker addr: tcp://127.0.0.1:37763 name: 4
Starting worker compute stream, tcp://127.0.0.1:37763
Register worker addr: tcp://127.0.0.1:40125 name: 3
Starting worker compute stream, tcp

# RAMS

In [3]:
rams_ds = ps.icons.read_full_rams_output("ic_wk")
# Limit the data variables
rams_ds = cd.rams.calculate_bsr_variables(
    ds=rams_ds, base_state=rams_ds.isel({"time": 0}, drop=False)
)
rams_ds = rams_ds[["UC", "VC", "WC", "THETA_bsr"]]
rams_ds

AttributeError: module 'ps' has no attribute 'icons'

In [ ]:
# Just pick t=100 minutes as the sample
computed_ds = rams_ds.sel({"time": ps.icons.to_sim_time(100)}).compute()
computed_ds.to_netcdf(
    "/home/cmdavis4/projects/cloudy/examples/example_data/pv_example_rams_data.nc"
)

# Trajectories

In [3]:
trajectory_ds = xr.open_dataset(
    "/moonbow/cmdavis4/projects/bl_transport/rams_io/isolated_convection_wk/all_tracers/newtracer_forcing-warmbubble4_deltax-500_frqlite-2_nobulku/trajectories/thesisplus_ft.nc"
)
trajectory_ds

<xarray.Dataset> Size: 72GB
Dimensions:              (time: 781, parcel_ix: 673845)
Coordinates:
  * time                 (time) datetime64[ns] 6kB 1991-04-26T21:50:00 ... 19...
  * parcel_ix            (parcel_ix) int64 5MB 449391 469291 ... 254916 288879
    t_minutes            (time) float64 6kB ...
    rs                   (parcel_ix) bool 674kB ...
Data variables: (12/25)
    x_ix                 (time, parcel_ix) float64 4GB ...
    x                    (time, parcel_ix) float64 4GB ...
    y_ix                 (time, parcel_ix) float64 4GB ...
    y                    (time, parcel_ix) float64 4GB ...
    z_ix                 (time, parcel_ix) float64 4GB ...
    z                    (time, parcel_ix) float64 4GB ...
    ...                   ...
    initial_y            (parcel_ix) float64 5MB ...
    group                (parcel_ix) <U9 24MB ...
    initialization_time  (parcel_ix) datetime64[ns] 5MB ...
    lofted               (parcel_ix) bool 674kB ...
    in_analysis          (parcel_ix) bool 674kB ...
    time_of_entrainment  (parcel_ix) datetime64[ns] 5MB ...

In [4]:
lofted_parcel_ixs = trajectory_ds["parcel_ix"].values[trajectory_ds["lofted"]]
lofted_ds = trajectory_ds[["x", "y", "z", "initialization_time", "THETA"]].sel(
    {"parcel_ix": cd.utils.maybe_random_choice(lofted_parcel_ixs, 25)}
)
lofted_ds = lofted_ds.compute()
lofted_ds

<xarray.Dataset> Size: 638kB
Dimensions:              (time: 781, parcel_ix: 25)
Coordinates:
  * time                 (time) datetime64[ns] 6kB 1991-04-26T21:50:00 ... 19...
  * parcel_ix            (parcel_ix) int64 200B 96626 78620 ... 229339 61422
    t_minutes            (time) float64 6kB 50.0 50.0 50.0 ... 179.0 179.0 180.0
    rs                   (parcel_ix) bool 25B False False True ... True False
Data variables:
    x                    (time, parcel_ix) float64 156kB nan nan ... 1.6e+05
    y                    (time, parcel_ix) float64 156kB nan nan ... 1.401e+05
    z                    (time, parcel_ix) float64 156kB nan nan ... 1.254e+04
    initialization_time  (parcel_ix) datetime64[ns] 200B 1991-04-26T22:10:00 ...
    THETA                (time, parcel_ix) float64 156kB nan nan ... 338.8 346.4

In [5]:
lofted_ds.to_netcdf(
    "/home/cmdavis4/projects/cloudy/examples/example_data/pv_example_trajectory_data.nc"
)